# TaskManager v2b：有边界的状态与生命周期离线重放

本记录属于开发集重复回归，不是独立正式测试集结果。输入是已冻结的 v2b 单轮 LLM 候选；未修改冻结的 `.san` 文件，也未再次调用 LLM。候选快照：`data/experiments/snapshots/formal_30_v4_task_manager_semantic_probe_v2b_result_20261028_181549/`。源 SSM 来自 `formal_30_v4_semantic_focus_v2_TaskManager_automatic_fail_20261028_140129/repair_history.json`。


## 诊断、规则与边界

候选把源 Vue `mounted` 中的首个 `loadInitialData()` 放到了 San `inited` 的末尾，而后两个调用从 `attached` 开始。仅当 SSM 明确映射 `mounted → attached`、首两个调用均为无参顶层调用、首调用是 `inited` 最后一条语句且第二调用是 `attached` 第一条语句时，才把首调用移回 `attached`。

候选 `moveTask` 对从 `this.data.get('tasks')` 得到的别名直接执行两次 `splice`，没有写回。SSM `methods[].body` 仅有前 500 字符，无法看到修改；完整证据在 `body_full`。数组规则要求源方法与候选方法的 `splice` 数量一致、仅一个被修改的状态数组别名、所有修改处于同一条件块、没有提前退出或已有写回，然后克隆数组并在最后一次修改后写回。其他不确定情形保持不变。另保留先前有边界的未解析源事件别名修正。


## 离线结果与未解决项

静态错误码由 `event_arguments_not_landed`、`lifecycle_behavior_not_landed`、`missing_event_bindings`、`missing_event_handlers`、`nested_state_mutation` 缩减为事件相关三项：`event_arguments_not_landed`、`missing_event_bindings`、`missing_event_handlers`。重放代码通过 San 编译；完整回归为 189/189，组合目标测试为 22/22。自动通过：否；人工通过：未验证。

仍须诊断事件绑定与实参语义，不可把本次离线改写计作自动迁移成功。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_v2b_bounded_state_lifecycle_replay.json`。


## 事件剩余项：v3b 单轮探针预运行冻结

对照 SSM 源事件与 v2b 候选，发现 22 处可疑 `handleX` 别名绑定，但别名存在不等于行为等价；剩余静态错误为 `missing_event_handlers`、`missing_event_bindings`、`event_arguments_not_landed`，其中参数例子为 `onStatClick(stat)`、`setFilter(filter.value)`。因此不采用批量确定性改名消除错误，而加入事件专用 repair prompt：包含源事件表达式与 `body_full` 方法代码，要求逐项核对行为、参数和子组件事件转发。

已冻结 `data/experiments/snapshots/formal_30_v4_task_manager_event_focus_v3b_pre_run_20261028/`，19 个 payload 校验通过；输入 SHA-256 `4f648583ab53bc279cc75862bf3aa677b6d3dd4aaf354e57c47c7aaadc7291b3`，prompt SHA-256 `33d70c86b694e7aec439b9cc2c5d80b36a0bc55eb284e941759ac9b92388c631`，文件与字符串哈希一致。此前 v3 预冻结因 Windows 换行转换造成两种哈希不一致，保留为被 v3b 取代的证据，不作为运行输入。当前没有模型请求，也没有人工验证；探针运行结果待续。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_event_focus_v3b_pre_run.json`。


## v3b 事件探针结果：静态失败、视觉通过、门禁拒绝

模型输出已冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_event_focus_v3b_result_20261028_203015/`，12 个 payload 的 SHA-256 校验通过，候选 SHA-256 为 `8866bc0ea6d445e90125ee7d01dec3aa852bdccfb99b1e79045a84e519e97449`。模型 qwen-plus 总计 24,760 tokens。

事件覆盖从 2 个精确绑定 + 22 个可疑别名变为 24 个精确绑定 + 0 个别名，源事件参数也恢复了。但模型把设置按钮与设置面板关闭操作改为三处非法 `on-click="showSettings = true/false"`（候选第 13、58、62 行），并移除了此前有效的设置 handler。静态校验失败：`invalid_san_event_handler_syntax`；视觉评估通过、结构相似度 0.9889；事件回归门禁 `reject`，自动通过为否，未人工验证。

该问题不是校验误报，也不能只靠方法改名宣称通过。下一步候选方案是对源 SSM 可证实的顶层布尔 data 赋值事件尝试定界的 repair 阶段转换；须先有反例测试与离线重放，再决定是否再次调用模型。本轮未修改冻结候选。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_event_focus_v3b_result.json`。


## 布尔赋值事件的定界规则与离线重放

针对 v3b 的三处非法设置事件，新增 repair 阶段通用定界规则：仅当 SSM 事件表达式精确包含 `field = true/false`，事件类型及出现次数一致，`field` 是源 data 且候选父组件在 `dataTypes` 和 `initData` 中声明布尔值时，才将外层 San 模板绑定改成命名方法，并在该方法中通过 `this.data.set(field, literal)` 写回。若已存在唯一的等价 setter，则复用；未知表达式、子组件模板、字段/类型不符、重复次数不符或命名冲突均不改。规则同时进入确定性 repair 分支和模型响应后处理。

冻结的 v3b 失败候选离线重放结果：静态错误码从 `invalid_san_event_handler_syntax` 变为空；San 编译通过、视觉评估通过（结构相似度 0.9889）、事件门禁 `pass`，完整测试 193/193 通过。快照：`data/experiments/snapshots/formal_30_v4_task_manager_event_boolean_offline_pass_20261028_210032/`，12 个 payload 校验通过，候选 SHA-256 `fdf553870bb55ec8386e00aa13c87cd42af95e5fb9630dbe4d788cbe3f3786f7`。未调用模型、未修改原冻结候选、未做浏览器人工验证；这不是新一轮自动批处理成功，更不是独立测试集结果。机器可读摘要：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_boolean_event_offline_replay.json`。


## 下一轮开发集回归预冻结（尚未运行）

新建单样本配置 `migration_pipeline/batch_configs/formal_30_v4_task_manager_boolean_event_regression_v1.json`，仅包含冻结的 `TaskManager.vue`，修复上限 3 轮，结果使用独立目录及 summary。dry-run 通过。预运行快照 `data/experiments/snapshots/formal_30_v4_task_manager_boolean_event_regression_v1_pre_run_20261028/` 已冻结，28 个 payload 校验通过，输出路径冻结时为空，未调用模型。源 SHA-256 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c`。

配置原始文件 SHA-256 为 `10d94bc9fd9049e41f420f44fc4c3c0e1486c51a9a4d551b0967305da6462193`；批处理 checkpoint 使用规范化 JSON 指纹 `6532a91c4d4e5d93600e2e28792439aef159b1c32562b18828409bfa0dd8bcb8`，两者的哈希对象不同。完整测试 193/193 通过。此回归复用开发源，不属于独立正式测试集；离线修复候选只是前置证据，不是本批次输入。结果待用户运行后追加。机器可读摘要：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_boolean_event_regression_v1_pre_run.json`。


## 单样本开发回归结果：布尔事件规则生效，但整轮仍失败

2026-10-28 的 `formal_30_v4_task_manager_boolean_event_regression_v1` 已运行并冻结：`data/experiments/snapshots/formal_30_v4_task_manager_boolean_event_regression_v1_TaskManager_automatic_fail_20261028_220158/`。11 个 payload 的 SHA-256 已校验。该样本是重复使用的开发集，不是独立测试集。

批次 1/1 完成、0/1 自动通过，停止原因为 `max_repair_rounds_reached`；三轮修复依次为两轮确定性修复和一轮 qwen-plus。生成 97,255 tokens，修复 121,511 tokens，总计 218,766。第二轮 `san_source_boolean_event_normalizer` 确实将三处 `showSettings = true/false` 事件赋值替换为处理函数，因此不能把本次失败归因于该规则未触发。

最终视觉评估通过，结构相似度 0.9889，但静态校验仍有 7 条错误：`unsupported_san_inline_template_math_global`、`invalid_san_event_handler_syntax`、`invalid_san_init_data_store_access`、`invalid_san_computed_instance_method_call`、两条 `hidden_template_reactive_dependency`、`lifecycle_behavior_not_landed`。其中事件语法残留是两处 `on-*.prevent`，不属于布尔赋值规则的处理范围。未对本次生成物进行浏览器人工验证。

结论：既不能把先前离线重放通过算作新一轮迁移通过，也不能因视觉通过而忽略静态失败。下一步应按错误族诊断、设计有界修复和反例测试，再运行单样本回归；不要直接改动冻结的 `.san` 结果。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_boolean_event_regression_v1_result.json`。


## 失败后：两个有界规则的离线重放

针对本轮冻结的失败结果，扩展了 repair 阶段的两个通用有界规则：一是把空值 `on-*.prevent=""` / `on-*.stop=""` 转为 San 的值修饰符与组件内空处理方法，不改非空 handler；二是仅在 SSM 源 `mounted` 完整调用序列与 `inited` 末尾调用逐项一致、且 `attached` 为空时，按原顺序移动整组调用。均加入不匹配即不改的反例测试。

未改动冻结生成物，也未调用 LLM。离线重放快照：`data/experiments/snapshots/formal_30_v4_task_manager_semantic_residual_bounded_replay_20261028_224401/`，9 个 payload 的 SHA-256 已校验。静态错误从 7 条减至 5 条；剩余为内联模板 `Math`、`initData` 访问 data、computed 调用实例方法、两条隐藏响应式依赖。San 编译和视觉评估通过，结构相似度 0.9889；完整测试 197/197 通过。

本次只是开发集部分离线修复，不是自动批处理通过，也不是独立测试结果；浏览器人工验证未做。下一步可在该冻结候选上进行一次受控的语义修复探针，先冻结输入与 prompt，再由用户启动 LLM 请求。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_semantic_residual_bounded_replay.json`。


## 五条剩余语义错误：单次修复探针预运行冻结

探针脚本 `migration_pipeline/probe_formal_30_v4_task_manager_semantic_residual_v1.py` 以已冻结的部分离线候选为输入，确认当前不再触发确定性修复分支，且激活已有的集中语义修复 prompt；这里只准备一次模型修复请求，不重新生成 Vue→San，也不接触独立测试集。

预运行快照：`data/experiments/snapshots/formal_30_v4_task_manager_semantic_residual_probe_v1_pre_run_20261028/`，18 个 payload 的 SHA-256 校验通过；输入 SHA-256 `fd759f5ffb2c751a85064f768f7e8a9177989cf88d625afc539fd86b6da9b205`，prompt SHA-256 `a585acb439fa19c908e29f03cda79c452c8a4f63b4a0846c616e907fe33e0d67`。冻结后重新校验了当前输入、prompt 和代码哈希。尚未发送模型请求，结果待用户执行 `--execute` 后检查并冻结。


## 五条剩余错误的单次 LLM 探针：由 5 条降至 2 条，仍未通过

用户运行了 `probe_formal_30_v4_task_manager_semantic_residual_v1.py --execute`。结果已在诊断前冻结：`data/experiments/snapshots/formal_30_v4_task_manager_semantic_residual_probe_v1_result_20261028_230247/`，12 个 payload 的 SHA-256 校验通过。模型为 qwen-plus，本次 prompt 12,521 tokens、completion 10,792 tokens、总计 23,313 tokens。

静态错误由 5 条降为 2 条：模型把内联模板 `Math.abs(trend)` 移入子组件方法，并为 `getTaskCount` 和 `isTagSelected` 显式传入响应式字段；但 `TaskForm.initData()` 仍在数据存储初始化前调用 `this.data.get('task')`，父组件 `filteredTasks` computed 仍调用实例方法 `this.isOverdue(task)`。视觉评估通过，结构相似度 0.9889，事件回归门禁 pass；自动通过为 false，未做浏览器人工验证。

另发现模型新增了 `inited()` 中的 `this.data.set('tasks', this.getSampleTasks())`，位于任务 watcher 注册之后、`attached()` 调用 `loadInitialData()` 之前。源 Vue 的 `tasks` 初始值是空数组，仅在 `mounted()` 中加载持久化或样本任务；这笔提前写入并非源行为，可能触发自动保存，须视为新的语义风险，不可因视觉通过忽略。下一步应针对这两条静态错误和提前写入设计受控修复与回归门禁，再考虑新一轮复跑。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_semantic_residual_probe_v1_result.json`。


## 两条残留错误与提前写入回归：收窄 prompt 和门禁

针对上一探针的失败，在 repair prompt 中新增仅适用于 `invalid_san_init_data_store_access` 与 `invalid_san_computed_instance_method_call` 同时存在、且无视觉错误的双错误路径：提供相关源子组件、源方法和生命周期上下文；要求把依赖 prop 的初始状态移到 `inited()`，将 computed 所需的纯计算提取为独立函数，同时保持模板、事件和其他行为不变。探针额外明确要求去除父组件提前写入 `tasks`。

通用回归门禁也扩展为：若源 Vue 仅在 `mounted()` 调用的加载方法写入某父组件字段，San 父组件 `inited()` 却写入该字段，则即使这是输入候选中已有的写入，也标记 review 并阻止自动通过。对上一冻结探针的事后重评为 `review`、字段 `tasks`；原冻结报告中的旧事件门禁 `pass` 保留不改。门禁对源 `created` 存在、嵌套回调写入和不属于 mounted 加载的 prop 派生初始化保守跳过。

新单次模型探针的预运行快照：`data/experiments/snapshots/formal_30_v4_task_manager_initdata_computed_probe_v1_pre_run_20261028/`，19 个 payload 校验通过；输入 SHA-256 `bdfdf4114a1d154afc137806e29d6d557336bb2f6d69dc0052916e8aa8242ec0`，prompt SHA-256 `7a8690eadd4bc6a69e030b86f28791d110472bbc8a9b492934bf451a4acd5925`。dry-run 与冻结后重校验通过，完整测试 202/202 通过；尚未请求模型，也未作浏览器人工验证。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_initdata_computed_probe_v1_pre_run.json`。


## 双错误单次探针结果：自动校验通过，语义审计仍需处理

用户于 2026-10-28 执行 `probe_formal_30_v4_task_manager_initdata_computed_v1.py --execute`。结果先冻结在 `data/experiments/snapshots/formal_30_v4_task_manager_initdata_computed_probe_v1_result_20261028_233258/`，12 个 payload 的 SHA-256 已校验；预运行输入与 prompt 哈希均匹配。qwen-plus 本次消耗 prompt 12,448、completion 10,818、合计 23,266 tokens，无 pipeline 后处理。

报告的静态验证与视觉评估均通过，静态复核仍为 0 条错误，结构相似度 0.9889，回归门禁 `pass`。差异核对确认：`TaskForm.initData()` 不再访问尚未初始化的 `this.data`；父组件 computed 改用独立纯函数 `isOverduePure`；父组件 `inited()` 中提前写入 `tasks` 的代码已删除，`attached()` 仍调用 `loadInitialData()`。因此本次探针报告 `automatic_passed=true`。

但源码 Vue 的 `TaskForm.data()` 在首次创建时以 `this.createFormData(this.task)` 初始化表单。新 San 候选仅在 `initData()` 返回空白默认表单，并在 `inited()` 注册 `task` watcher，**没有用初始 `task` prop 初始化 `formData`**。独立的只读 Node/San 实例检查传入初始 `task.title='existing'` 后，`formData.title` 仍为空字符串，确认 watcher 不会代替首次初始化。因此编辑弹窗首次挂载会丢失原任务字段。此项不由当前静态、视觉或回归门禁证明通过。故研究结论记为 `review_required`，不把报告中的自动通过等同于语义通过，也不计作独立正式测试成功。后续需补有界的 prop 派生初始状态校验/修复，再做新的开发集整轮回归和人工编辑场景验证。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_initdata_computed_probe_v1_result.json`。


## 首次 prop 派生状态：新增有界校验与单错误修复探针

针对上一轮 `TaskForm` 编辑态初始值丢失，新增源代码感知的 `missing_inline_prop_dependent_initialization` 校验：仅在 Vue 源码中存在具名内联子组件、声明 props，且 `data()` 某字段的初始表达式读取 `this.prop`，并能匹配同名 San 内联组件时，要求 San 子组件 `inited()` 在注册 watcher 前显式读取初始 prop 并写入对应字段。其他源码形态保持未评估，不凭猜测判失败。冻结的旧结果未修改；该规则也揭示了历史 `MonitoringDashboard` v1 候选的同类缺陷，相关回归测试已更新为如实表达历史结果。

为仅剩此错误的情况增加聚焦 repair prompt，携带对应 Vue 内联子组件源码，并明确 watcher 不会替代首次初始化。定向和完整测试为 207/207 通过。预运行输入、prompt、实现及错误列表已冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_inline_prop_probe_v1_pre_run_20261028/`，18 个 payload 的 SHA-256 校验通过。输入 SHA-256 `6bd3a33b57723399ac8704525f214233661dd823b1eb404b0e592760624e1157`，prompt SHA-256 `02f9e57ae4ab3484a9e002fb4fa9c21f5a4e81cef4fdc85f2f6e96ad754716dd`。尚未发送模型请求或进行浏览器人工验证。此探针只复用开发集候选，不是新整轮批处理或独立测试证据。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_inline_prop_probe_v1_pre_run.json`。


## 首次 prop 派生状态单错误探针：局部修复通过

用户执行 `probe_formal_30_v4_task_manager_inline_prop_v1.py --execute` 后，先将原始结果冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_inline_prop_probe_v1_result_20261028_235822/`，12 个 payload 的 SHA-256 校验通过。输入、prompt 与预运行冻结哈希一致；qwen-plus 消耗 prompt 12,047、completion 10,840、合计 22,887 tokens，未做 pipeline 后处理。

与冻结输入相比，输出仅在 `TaskForm.inited()` 的 `task` watcher 之前增加两条语句：先 `this.data.get('task')`，再 `this.data.set('formData', this.createFormData(task))`。只读 San 实例复核：传入初始任务 `title='Existing'`、`priority='high'`、`tags=['urgent']` 时，旧候选表单标题为空；新候选保留上述字段，且标签数组为独立副本。独立重跑的静态验证、San 编译、视觉评估均通过，结构相似度 0.9889；事件与提前状态写入回归门禁 `pass`。

结论限定为复用开发集候选的一次局部修复探针通过。未做浏览器人工验证，也不是新的 Vue→San 整轮批处理或独立正式测试结果。下一步应冻结新一轮开发集批处理协议，再从源 Vue 重新迁移并检查是否仍发生相同缺陷。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_inline_prop_probe_v1_result.json`。


## 2026-10-29：从冻结源 Vue 重新迁移的单样本回归预运行

局部探针通过后，为验证规则是否在完整管线中生效，准备独立输出路径的开发集单样本批次 `formal_30_v4_task_manager_inline_prop_regression_v1`。输入仍是已冻结协议 `formal_30_v3_frozen_20261022` 中的 `TaskManager.vue`，源 SHA-256 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c`；不将探针修复的 San 候选当作批处理输入。配置 dry-run 通过，输出、修复报告和 summary 路径在冻结时均为空。

预运行快照：`data/experiments/snapshots/formal_30_v4_task_manager_inline_prop_regression_v1_pre_run_20261029/`；28 个 payload 的 SHA-256 校验通过，包含源 Vue、批次配置、管线/服务器关键代码、针对性测试及上一探针的清单。配置文件字节 SHA-256 `e74a7d5c2d9e3ac80d5c7ef4919b9e3d820b8fda8f5200e260a9deb17562e66e`。冻结前完整测试 207/207 通过。尚未运行模型批处理，也未做浏览器人工验收；这仍是重复使用的开发集样本，不是独立正式测试证据。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_inline_prop_regression_v1_pre_run.json`。


## 2026-10-29：首次 prop 规则完整管线回归，局部生效但批次仍失败

用户完成 `formal_30_v4_task_manager_inline_prop_regression_v1` 的源 Vue→San 单样本批处理。诊断前已冻结原始结果于 `data/experiments/snapshots/formal_30_v4_task_manager_inline_prop_regression_v1_TaskManager_automatic_fail_20261029_145142/`，11 个 payload 的 SHA-256 已校验，未覆盖预运行或先前探针。批次 1/1 完成、0/1 自动通过；两轮确定性修复加一轮 qwen-plus 修复后达到 3 轮上限。生成 97,580 tokens，修复 122,045 tokens，合计 219,625。

新规则确实被使用：初始生成物含 `missing_inline_prop_dependent_initialization`；最终 `TaskForm.inited()` 已在 `task` watcher 注册前读取初始 `task` 并经 `createFormData` 写入 `formData`，该错误消除。最终视觉评估通过，结构相似度 0.9889，但静态校验仍有三类错误：外层任务列表 `on-dragover=""`（源 Vue 为 `@dragover.prevent`）；父组件 `filteredTasks` computed 调用 `this.isOverdue(task)`；模板 `getTaskCount(filter.value)` 内隐式读取 `tasks`，导致响应式依赖不可见。故最终自动失败 `max_repair_rounds_reached`，没有进行浏览器人工验收，不可因视觉通过而计作完整迁移通过。

结论：首次 prop 派生状态修复在完整管线中落地，但还有三个独立残留。下一步应在该冻结失败产物上做有界的离线规则/聚焦修复实验，先冻结输入与 prompt，再请求模型；禁止直接覆盖失败 `.san` 或提高修复轮数掩盖缺陷。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_inline_prop_regression_v1_result.json`。


## 三类残留错误：聚焦修复探针预运行冻结

基于 2026-10-29 冻结的完整批次失败产物，仅针对 `invalid_san_event_handler_syntax`、`invalid_san_computed_instance_method_call`、`hidden_template_reactive_dependency` 增加一条聚焦 repair prompt 路由。要求修复源 Vue `@dragover.prevent` 对应的空 San handler、把 `isOverdue` 的纯逻辑移出 computed 实例方法调用、将模板 `getTaskCount` 所需 `tasks` 显式传参；保留已落地的 `TaskForm` 首次 prop 初始化、其他事件、生命周期与样式。针对性测试及完整测试 208/208 通过。

单次模型探针 `probe_formal_30_v4_task_manager_three_residual_v1.py` dry-run 确认冻结候选只剩上述三类错误，且不会走确定性修复分支。预运行快照 `data/experiments/snapshots/formal_30_v4_task_manager_three_residual_probe_v1_pre_run_20261029/` 含 19 个已校验 SHA-256 的 payload；输入哈希 `8f52867411bae34f05bfd6dc0704a0cfb21dbb26830592bc0c4c057cd590bbbb`，prompt 哈希 `a6d6746189cdc63ab7caaa54d6c65298842da635da04ad3c478298a9379a921a`。尚未请求模型，也未进行浏览器人工验证。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_three_residual_probe_v1_pre_run.json`。


## 2026-10-29 三残留单次探针：局部通过，待新整轮验证

用户运行 `probe_formal_30_v4_task_manager_three_residual_v1.py --execute` 后，先将原始结果冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_three_residual_probe_v1_result_20261029_153143/`，10 个 payload 的 SHA-256 校验通过，输入与 prompt 均匹配预运行哈希。qwen-plus 消耗 prompt 12,745、completion 11,118、合计 23,863 tokens；无 pipeline 后处理。

差异核对显示：空 `on-dragover` 改为 `prevent:noop` 并定义父组件 no-op 方法；`isOverdue` 原计算移到脚本级纯函数，computed 与统计更新调用该函数，实例方法保留委托；模板 `getTaskCount(tasks, filter.value)` 显式传入响应式 `tasks`，对应方法不再隐藏读取它。未改动 `TaskForm` 首次 prop 初始化、其他组件定义或样式。独立重跑静态验证 0 错、San 编译通过、视觉评估通过，结构相似度 0.9889；回归门禁 `pass`。只读 San 实例检查确认首次编辑标题和标签副本保留，示例任务的 all/active/urgent/completed 计数分别为 2/1/1/1。

结论仅为冻结开发集失败候选上的单次局部修复探针通过；未做浏览器人工验收，也不替代新一轮从源 Vue 开始的完整批处理，更不是独立正式测试证据。下一步应先冻结新的开发集单样本回归，再检验完整管线是否收敛。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_three_residual_probe_v1_result.json`。


## 2026-10-29：三残留修复后完整管线回归预运行冻结

局部探针通过后，为检验完整源 Vue→San 管线是否在 3 轮修复限制内收敛，准备开发集单样本批次 `formal_30_v4_task_manager_three_residual_regression_v1`。输入仍是冻结协议中的 `TaskManager.vue`，源 SHA-256 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c`；不将探针生成的 `.san` 当作批处理输入。配置 dry-run 通过，所有结果输出路径在冻结时为空。

预运行快照 `data/experiments/snapshots/formal_30_v4_task_manager_three_residual_regression_v1_pre_run_20261029/` 包含 29 个已校验 SHA-256 的 payload：源 Vue、批次配置、关键实现及测试、上一次局部探针清单。配置文件字节 SHA-256 `524e65f449e17b91c2573894371aa04f79f1f0d2f96eb2f680a2086e40b422e2`。完整测试 208/208 通过。尚未请求模型，也没有浏览器人工验收；该样本仍属重复使用的开发集，不是独立正式测试证据。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_three_residual_regression_v1_pre_run.json`。


## 2026-10-29 新整轮回归：事件门禁拒绝，不能计为通过

用户完成 `formal_30_v4_task_manager_three_residual_regression_v1` 单样本开发集回归。诊断前将原始结果冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_three_residual_regression_v1_TaskManager_automatic_fail_20261029_155522/`，11 个 payload 的 SHA-256 校验通过。批次 1/1 完成、0/1 自动通过；修复第 1 轮为确定性规则，第 2、3 轮为 qwen-plus。生成 97,216 tokens，修复 144,863 tokens，合计 242,079。停止原因为 `repair_event_contract_regression`，不是自动通过。

错误轨迹：初始生成物 14 类静态错误，第 1 轮后 11 类，第 2 轮后剩 `invalid_san_event_handler_syntax` 与 `invalid_san_computed_instance_method_call`；第 3 轮后剩前者一类。`TaskForm` 首次 prop 初始化已在第 2 轮落地。第 3 轮保留了任务列表空 `on-dragover=""`，与源 Vue 的 `@dragover.prevent` 不一致；同时把模态框外层 `on-click="closeModal"` 改为 `handleModalOverlayClick` 并添加内层 stop handler。虽然新方法委托给 `closeModal`，当前事件契约门禁无法证明其与源 `@click.self` 的等价性，因此发现一条新增未解析的源绑定并判 `reject`。这不等于已证明浏览器行为错误，但必须阻止自动放行。

最终视觉评估通过，结构相似度 0.9611；静态校验仍失败，交互评估与浏览器人工验收未执行。不能用视觉通过掩盖静态失败或事件门禁拒绝。下一步应从第 2 轮冻结候选做有界修复，明确保留源命名的事件契约，优先处理空 dragover 和 computed 调用，不直接改动冻结输出。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_three_residual_regression_v1_result.json`。


## 2026-10-29：TaskManager 两类残留错误的受约束探针预运行冻结

从已冻结的失败批次 `formal_30_v4_task_manager_three_residual_regression_v1_TaskManager_automatic_fail_20261029_155522` 提取第 2 轮候选，不修改原始生成文件。静态复核仅剩 `invalid_san_event_handler_syntax` 和 `invalid_san_computed_instance_method_call`；现有确定性修复不会改写该候选。源 Vue 任务列表使用 `@dragover.prevent`，模态框使用 `@click.self`。本探针仅允许补齐空的 dragover 处理器，并把 computed 对实例 `isOverdue` 的调用改为等价纯函数；明确禁止改写其他事件绑定、模态框处理器和 DOM/CSS。

预运行快照：`data/experiments/snapshots/formal_30_v4_task_manager_event_computed_probe_v1_pre_run_20261029/`，20 个 payload 的 SHA-256 校验通过。输入 SHA-256 `061da1a253123d3e1065b7d013d5c8870738fa12233e0085fe60d7235c68ed3c`；Prompt SHA-256 `628b4afa0176800d36d708c5e8be0498917115309d178c7b3cfffc1eccaacfc0`。本地 209 项测试通过。尚未请求 LLM，也未进行浏览器人工验收；本探针只属于重复使用的开发集，不是独立测试证据。


## 2026-10-29：两类残留错误受约束探针结果

用户完成 `probe_formal_30_v4_task_manager_event_computed_v1.py --execute` 后，先冻结原始产物于 `data/experiments/snapshots/formal_30_v4_task_manager_event_computed_probe_v1_result_20261029_164017/`，11 个 payload 的 SHA-256 校验通过，输入及 Prompt 哈希与预运行冻结一致。qwen-plus 使用 22,940 tokens。

局部探针的静态验证无错误、视觉评估通过（结构相似度 0.9611）、事件契约门禁 `pass`，因此仅该探针的 `automatic_passed=true`。差异核对显示空 `on-dragover` 改为 `prevent:noop` 并增加父组件 no-op；新增脚本级 `isOverdueHelper` 替代 computed 等处的实例方法调用；模态框既有事件绑定未改动。原 `isOverdue` 方法和 helper 目前保留相同计算体，而非委托关系，后续如修改算法需防止漂移。事件门禁也不证明原有模态框 `@click.self` 的运行时等价性。

本结果只证明在冻结的第 2 轮候选上的局部修复可行；尚未完成从源 Vue 开始的新鲜完整批处理回归，也未做浏览器人工验收，不属于独立测试集。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_event_computed_probe_v1_result.json`。下一步是准备并运行单样本完整开发回归，先冻结结果，再决定是否进入人工验证。


## 2026-10-29：TaskManager 完整管线开发回归预运行冻结

局部受约束探针通过后，准备从冻结源 Vue 重新运行单样本完整管线批次 `formal_30_v4_task_manager_event_computed_regression_v1`，不以探针修复出的 San 作为输入。配置 dry-run 通过，预运行快照位于 `data/experiments/snapshots/formal_30_v4_task_manager_event_computed_regression_v1_pre_run_20261029/`，31 个 payload 的 SHA-256 校验通过。配置 SHA-256 为 `16d4f92690f5be5cb8fac3f895e61a21d8a4b9a0e83225c2e933d150c9306464`，源 Vue SHA-256 为 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c`。输出路径冻结时为空；尚未发起模型请求。

本回归仍使用开发集旧样本，不能当作独立测试证据。批次执行后须先冻结原始结果，再区分自动校验与人工验收。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_event_computed_regression_v1_pre_run.json`。


## 2026-10-29：两类残留探针通过后的完整管线回归失败

用户完成 `formal_30_v4_task_manager_event_computed_regression_v1` 单样本开发回归后，先冻结原始结果于 `data/experiments/snapshots/formal_30_v4_task_manager_event_computed_regression_v1_TaskManager_automatic_fail_20261029_170150/`，11 个 payload 的 SHA-256 校验通过。批次 1/1 完成、0/1 自动通过，修复轮次依次为确定性、确定性、qwen-plus；生成 97,308 tokens，修复 121,572 tokens，合计 218,880。停止原因为 `max_repair_rounds_reached`。

初始生成物有 14 类静态错误；前两轮确定性修复后仍有 12 类，第三轮模型修复后剩 5 类：`invalid_san_init_data_store_access`、`missing_inline_prop_dependent_initialization`、`invalid_san_computed_instance_method_call`、`hidden_template_reactive_dependency`、`lifecycle_behavior_not_landed`。具体包括 TaskForm 在 `initData` 中读取尚未初始化的 `this.data`、首个 task prop 未正确导入 formData、computed 调用实例 `isOverdue`、模板计数方法隐藏读取 tasks，以及源 mounted 行为被提前放入 inited。该轮模型收到的是广义修复 Prompt；此前新增的两类错误专用 Prompt 因输入仍有 12 类错误而没有触发。

最终视觉评估通过，结构相似度 0.9889，但静态校验失败；交互评估与浏览器人工验收未进行。不能把局部探针通过解释为完整管线通过，也不能用视觉通过覆盖静态失败。下一步应基于冻结的第 3 轮候选做针对 5 类残留的受约束单次探针，核对源 Vue 生命周期与 TaskForm 初始 prop 语义；不直接修改冻结产物，不盲目增加修复轮数。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_event_computed_regression_v1_result.json`。


## 2026-10-29：TaskManager 五类残留错误单次探针预运行冻结

从已冻结失败批次 `formal_30_v4_task_manager_event_computed_regression_v1_TaskManager_automatic_fail_20261029_170150` 的第 3 轮 San 候选出发，静态复核恰好有 5 类残留错误，且现有确定性修复不会再改写该候选。对照源 Vue 确认：TaskForm 的 data() 用首次 task prop 创建 formData，并监听后续变化；父组件 mounted() 依次调用 loadInitialData、startAutoSync、showNotification。新专用 Prompt 要求保持这两项语义，并修复 computed 实例方法调用及模板计数的隐藏响应式依赖，不改动其他事件、DOM 和样式。

预运行快照：`data/experiments/snapshots/formal_30_v4_task_manager_five_residual_probe_v1_pre_run_20261029/`，20 个 payload 的 SHA-256 校验通过。输入 SHA-256 `f82e96741369a8a9339654d0a6bd2b7370b2abe8bb0b4890a12d4ef3aa3c5baa`；Prompt SHA-256 `43811c2701929289ca83b2f12efc2bc6db82e1103c541121648bb648e4e26abb`。本地 210 项测试通过。尚未调用模型，也未进行浏览器人工验收。本探针属于复用开发样本，不是独立测试证据。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_five_residual_probe_v1_pre_run.json`。


## 2026-10-29：TaskManager 五类残留错误单次探针结果

用户运行 `probe_formal_30_v4_task_manager_five_residual_v1.py --execute` 后，先冻结原始产物于 `data/experiments/snapshots/formal_30_v4_task_manager_five_residual_probe_v1_result_20261029_192110/`，11 个 payload 的 SHA-256 校验通过，输入及 Prompt 哈希与预运行冻结一致。qwen-plus 使用 24,221 tokens。

局部探针静态验证 0 错误，视觉评估通过（结构相似度 0.9889），事件契约门禁 `pass`。差异核对显示：TaskForm 的 initData 改为无 this.data 的默认状态，inited 导入首次 task prop 并监听后续变化；父组件启动调用按源 mounted 顺序移入 attached；纯 isOverdue helper 被 computed 调用，实例方法委托给 helper；模板将 tasks 显式传给 getTaskCount。模态框与拖拽事件绑定未更名。

结论仅为冻结的第 3 轮候选上的单次局部修复通过，不能代替新鲜完整管线回归或独立测试。浏览器人工验收尚未执行，事件门禁亦不证明所有运行时行为。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_five_residual_probe_v1_result.json`。下一步可依据前轮 2 次确定性修复、1 次广义模型修复后恰剩 5 类错误的事实，在新开发批次中有边界地允许第 4 轮专用修复 Prompt，再从冻结源 Vue 完整复跑。


## 2026-10-29：五类残留局部探针后完整管线回归预运行冻结

为检验局部探针是否可在新生成结果上复现，准备从冻结源 Vue 重新运行开发集单样本批次 `formal_30_v4_task_manager_five_residual_regression_v1`，最多 4 轮修复。第 4 轮限额的依据是上次完整批次前两轮为确定性规则、第三轮广义模型修复后恰余 5 类静态错误，而冻结的局部探针证明该错误组合可由一次专用请求修复；这不保证新批次有相同路径，也不豁免任何静态、视觉或事件门禁。

配置 dry-run 通过。预运行快照 `data/experiments/snapshots/formal_30_v4_task_manager_five_residual_regression_v1_pre_run_20261029/` 包含 31 个 SHA-256 校验通过的 payload；配置字节 SHA-256 `94f33339190c2308f05f74102b05d768a781bc4ef5e7648b1d5bd2e19b28ca52`，源 Vue SHA-256 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c`。输出路径在冻结时为空，尚未调用模型。复用开发集样本，不属于独立测试证据；局部探针产物只作证据，不作批次输入。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_five_residual_regression_v1_pre_run.json`。


## 2026-10-29：TaskManager 四轮上限完整管线开发回归自动通过

用户完成批次 `formal_30_v4_task_manager_five_residual_regression_v1` 后，先将原始 summary、repair history、源 Vue 及各轮 San 产物冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_five_residual_regression_v1_TaskManager_automatic_pass_20261029_200650/`；12 个 payload 的 SHA-256 校验通过。1/1 样本完成且自动通过，4 轮修复依次为确定性、确定性、qwen-plus、qwen-plus；生成 97,276 tokens，修复 145,043 tokens，合计 242,319。最终静态校验 0 错误，视觉评估通过，结构相似度 0.9889，事件契约回归门禁 `pass`，停止原因为 `passed`。

需要严格区分因果归因：第 3 轮后仅剩 `invalid_san_computed_instance_method_call` 和两处 `hidden_template_reactive_dependency`；第 4 轮实际使用已有的多语义残留 Prompt，并非新加的精确五类错误专用 Prompt。因此本结果说明有上限的第 4 轮在这一复用开发样本上收敛，不能声称五类错误专用 Prompt 在新鲜批次中被触发或证明其一般效果。

交互评估仍为空，浏览器人工验收未执行；自动通过不等于运行时行为与源 Vue 完全等价，也不属于独立测试集。下一步应针对本次冻结的最终 San 运行交互场景并与源 Vue 人工对照，再记录人工结论。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_five_residual_regression_v1_result.json`。


## 2026-10-29：TaskManager 冻结产物人工验证待确认

已将 `tests/manual/vue-test-runner.html` 的组件引用切换到本次自动通过快照的 `source.vue`，并将 `tests/manual/san-test-runner.html` 的组件引用切换到同一快照的 `repair_round_4.san`。仅修改测试组件引用与标签，未更改加载逻辑。此操作只是准备人工对照，不构成人工通过证据。优先检查初始任务、筛选计数、设置弹窗、任务创建和编辑的首个 prop 数据、标签状态、拖拽及自动同步；出现差异须先记录而不直接修改冻结产物。


## 2026-10-29：TaskManager 自动通过后的人工运行时失败

用户在本次冻结的自动通过产物上人工发现：点击设置弹窗或新建／编辑任务弹窗内部内容会意外关闭弹窗，且无法拖拽排序。此结论为用户人工观察，尚未附浏览器控制台或录屏；不能将此样本记为人工通过。自动批次的 `interaction_eval_passed` 为空，静态和视觉通过没有覆盖这两类运行时事件语义。

源 Vue 的两处弹窗外层分别使用 `@click.self`，仅外层背景本身被点击才应关闭；本次 San 却在外层直接使用 `on-click=closeModal` 和 `on-click=setShowSettingsFalse`，方法内部也没有 `event.target === event.currentTarget` 约束，内部点击冒泡时因此关窗。源 Vue TaskItem 使用原生 `@dragstart`、`@dragend`，再向父组件发出自定义 `drag-start`、`drag-end`；本次 San 的 TaskItem 原生 DOM 绑定却写成 `on-drag-start`、`on-drag-end`。此前人工通过的 San 版本使用 self-only 事件目标判断及原生 `on-dragstart`、`on-dragend`。

本次人工失败单独记录于 `data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_five_residual_regression_v1_manual_failure.json`，不覆盖冻结的自动通过产物。后续应增加源语义约束的通用验证／修复及交互回归，再由用户人工复核；不能仅复制旧组件修复作为通用效果证据。


## 2026-10-29：TaskManager 源事件语义门禁与离线受约束修复

针对冻结自动通过产物的人工失败，新增源 Vue 对照门禁：`@click.self` 要求 San 点击处理保留 `event.target === event.currentTarget` 保护；原生 `@dragstart/@dragend` 不得误写成 `on-drag-start/on-drag-end`，但子组件自定义 `drag-start/drag-end` 仍保留连字符。规则位于 `migration_pipeline/utils/source_event_semantics.py`，由 `stages/validate.py` 报错、`stages/repair.py` 在命中错误时执行受约束确定性修复。未根据组件名硬编码，缺少源 Vue 证据时不触发。

对冻结产物 `formal_30_v4_task_manager_five_residual_regression_v1_TaskManager_automatic_pass_20261029_200650` 只读重放：修复前命中 `missing_self_click_guard` 和 `native_drag_event_misbound`，修复后静态错误 0、San 编译通过、视觉评估通过，结构相似度 0.9889；完整 213 项单元测试通过。旧人工通过版本不触发上述两项。新候选冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_source_events_candidate_20261029_214742/`，SHA-256 校验通过，当前规则重放产物与冻结候选逐字节一致，原自动通过产物未修改。

这是复用开发样本的离线候选，不是独立测试集证据，也不是完整批处理自动通过的新结果。浏览器人工复核仍待完成；San 测试页已切换到新候选，Vue 仍引用同一冻结源。重点复核设置／新建／编辑弹窗内部点击、遮罩点击关闭，以及任务拖拽排序。


## 2026-10-29：TaskManager 源事件语义候选人工通过

用户在 `tests/manual/vue-test-runner.html` 与 `tests/manual/san-test-runner.html` 对照检查后确认新候选通过。验收范围是设置／新建／编辑弹窗内部点击不会意外关闭、点击遮罩可关闭，以及任务拖拽排序可用；逐场景浏览器日志或截图未提供，记录为用户确认，不能推断其他未检查交互。

人工通过证据单独冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_source_events_manual_pass_20261029_215547/`，候选 SHA-256 为 `f7e8eef6f66a2a46d6443548550f20207817ce8d75d1828118f36e6bb431c39f`，7 个 payload 的 SHA-256 已校验。离线候选的静态校验、San 编译和视觉评估此前均通过，完整 213 项测试通过。

证据性质：复用开发样本上的离线受约束修复和人工验收；不是新一轮完整管线自动回归，也不是独立测试集结论。先前自动通过产物与其人工失败记录均保持原状。后续可在锁定当前规则版本后复跑完整 TaskManager 开发回归，检验该门禁／修复是否会在生成流水线中触发；再决定是否进入独立测试。


## 2026-10-29：TaskManager 源事件语义完整管线开发回归预运行冻结

为检验此前离线修复是否进入完整生成流水线，新增独立批次 `formal_30_v4_task_manager_source_events_regression_v1`，仍只使用已冻结的 TaskManager 开发源样本，源 SHA-256 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c`。修复上限为 5 轮：前次完整路径使用 4 轮，最多额外容纳一次源事件语义的受约束修复；这不是通过保证。旧批次、离线候选及人工通过快照都只作谱系证据，不作为新批次生成输入。

配置位于 `migration_pipeline/batch_configs/formal_30_v4_task_manager_source_events_regression_v1.json`。预运行证据冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_source_events_regression_v1_pre_run_20261029/`，33 个 payload SHA-256 已校验；冻结时新输出目录、修复报告目录与 summary 均不存在。配置 dry-run 通过，完整 213 项 unittest 通过，尚未发起模型请求。该回归属于复用开发集，不是独立正式测试。运行时不要加 `--force`，完成后应先冻结原始结果，再判断自动通过和人工验收。


## 2026-10-29：源事件语义完整管线开发回归结果（自动失败）

用户完成批次 `formal_30_v4_task_manager_source_events_regression_v1`，原始 summary、repair history、源 Vue 与初始／五轮 San 先冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_source_events_regression_v1_TaskManager_automatic_fail_20261029_222317/`，13 个 payload SHA-256 校验通过。1/1 完成，0/1 自动通过，无执行错误；五轮上限耗尽。生成 97,543 tokens、修复 122,381 tokens、合计 219,924。交互评估为空，未对本次输出作浏览器人工验收。

五轮修复依次为：确定性结构修复、确定性布尔事件修复、确定性源事件语义修复、qwen-plus、确定性源事件语义修复。原生拖拽事件错误在第 3 轮后消失，弹窗 self-click 错误在第 5 轮后消失；说明新门禁与受约束修复在完整管线中实际触发，但两处修复分别消耗了轮次。最终视觉评估通过（结构相似度 0.9889），静态仍有四类错误：`invalid_san_init_data_store_access`、`missing_inline_prop_dependent_initialization`、`invalid_san_computed_instance_method_call`、`hidden_template_reactive_dependency`。不能把视觉通过或事件错误消失写成自动通过，也不能推断第 6 轮必然成功。

当前四类残留尚无精确组合的专用修复 Prompt，下一次若直接继续会走较宽的通用 Prompt。优先基于冻结的第 5 轮候选做有边界的四类残留单次开发探针，核对首次 prop 状态、computed 纯函数及模板显式响应依赖；若局部探针通过，再设计新的完整批次。机器可读记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_source_events_regression_v1_result.json`。


## 2026-10-29：TaskManager 四类残留错误单次探针预运行冻结

以已冻结的完整管线失败快照 `formal_30_v4_task_manager_source_events_regression_v1_TaskManager_automatic_fail_20261029_222317` 的第 5 轮 San 为唯一输入，新增精确四错误修复 Prompt。四类是 `invalid_san_init_data_store_access`、`missing_inline_prop_dependent_initialization`、`invalid_san_computed_instance_method_call`、`hidden_template_reactive_dependency`；提示词要求保留已修正的弹窗 self-click、原生拖拽、子组件自定义拖拽、DOM、CSS 与父级生命周期。提示词路由为通用错误组合，不以 TaskManager 组件名触发。

探针脚本 `migration_pipeline/probe_formal_30_v4_task_manager_four_residual_v1.py` 的 dry-run 证实上述四类错误恰好存在、当前确定性修复不再改写输入、精确提示词已触发。输入 SHA-256 `19d3bcd43e84020b21126f3dae898a094fa4149a6a191f4a0b0ff0102b878f49`，Prompt SHA-256 `36b23eeb12890faa1ba2250bcbc9927ada62e52b2b10ab89054228041e7eaa39`。完整 215 项 unittest 通过。预运行证据冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_four_residual_probe_v1_pre_run_20261029/`，21 个 payload SHA-256 校验通过；尚未发送模型请求。

该探针最多执行一次模型修复，不是完整新批次、不是独立测试集证据；返回后须先冻结原始结果，再核对静态、视觉与事件回归门禁，人工浏览器验收另计。


## 2026-10-29：四类残留单次探针结果（静态／视觉通过，回归门禁待审）

用户执行一次 qwen-plus 修复请求，使用 24,841 tokens。先将原始输入、提示词、候选和报告冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_four_residual_probe_v1_result_20261029_224507/`，11 个 payload 的 SHA-256 校验通过。候选 SHA-256 `6c74c46a62d0dda9bfc2275419cd25783d97a3d8db0420484a3257c6f7bdeac8`。四类静态错误均消失，San 静态／编译及视觉评估通过，结构相似度 0.9889；事件错误未新增，源事件未出现新增缺失。

但保守回归门禁判定 `review`，因此探针 `automatic_passed=false`。门禁指出父组件 `inited()` 在挂载前对 `settings`、`notification` 写入状态。前后父级 `inited()` 内容逐字一致，故这是探针输入已经存在的问题，不是本次四错误修复新引入的；门禁仍基于绝对生命周期风险阻止自动通过。源 Vue 在 data() 给出默认值，但 `loadInitialData()`、`startAutoSync()`、`showNotification()` 位于 mounted()；这些提前的同值写入可能触发 watcher，不能未经核查就忽略或把门禁改成通过。

下一步先针对父组件这两处提前写入和 watcher 副作用做有边界的源对照复核；若确认冗余，另建离线候选只删除相应 no-op get/set，复跑静态、编译、视觉、事件门禁，再作浏览器人工验收。此局部探针不属于新的完整管线自动通过或独立测试证据。机器记录：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261028/task_manager_four_residual_probe_v1_result.json`。


## 2026-10-29：TaskManager 挂载前冗余写入的离线受限候选

基于已冻结的四类残留单次探针候选 `formal_30_v4_task_manager_four_residual_probe_v1_result_20261029_224507`，仅从导出父组件 `inited()` 删除 `settings` 和 `notification` 各一对同值 get/set；保留 watcher、子组件、`attached()` 和事件逻辑。脚本：`migration_pipeline/replay_formal_30_v4_task_manager_premount_noop_fix.py`；三个边界测试通过，完整 218 项 unittest 通过。

新候选冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_premount_noop_candidate_20261029_225953/`，六个 payload 的 SHA-256 已校验。输入 SHA-256 `6c74c46a62d0dda9bfc2275419cd25783d97a3d8db0420484a3257c6f7bdeac8`，候选 SHA-256 `dd80e7039e21827a205f0974ae3517510a51a05b70d10f33559e3056807f524b`。静态验证、San 编译、可视化评估均通过，结构相似度 0.9889；事件回归门禁从 review 变为 pass，未发现新增源事件缺失或挂载前无依据写入。Vue 对照页源码与本次快照 `source.vue` SHA-256 相同。

这只是复用开发样本上的离线候选，不是完整批处理自动通过，也不是独立测试集证据。San 测试页已切换到此候选；浏览器人工验收待用户确认，重点是设置及新建／编辑弹窗内部点击、拖拽排序，以及初始通知／自动同步行为。


## 2026-10-29：挂载前冗余写入候选的人工验收

用户在 San/Vue 测试页对照后回复“通过”。该确认对应候选 SHA-256 `dd80e7039e21827a205f0974ae3517510a51a05b70d10f33559e3056807f524b`，此前要求复核的范围包括设置及新建／编辑弹窗内部点击、拖拽排序、初始通知与自动同步；未提供逐场景浏览器日志，因此仅记录用户确认，不伪造逐项结果。

人工通过证据冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_premount_noop_manual_pass_20261029_230328/`，七个 payload 的 SHA-256 已校验，包括候选、Vue 源码、离线复核报告和测试页。性质为复用开发集上的受限离线候选人工通过；完整批处理尚未对这一候选重跑，不能记为自动通过或独立测试集结果。后续需要把有依据的规则纳入修复链路，再单独运行完整开发回归。


## 2026-10-29：挂载前同值写入规则与完整开发回归预运行

鉴于 Prompt 已禁止将源 Vue `mounted()` 的写入提前，但此前完整管线仍产生父组件 `inited()` 的 `settings`／`notification` 同值回写，在 `migration_pipeline/stages/repair.py` 增加受限后处理规则：必须有 SSM 源 `mounted` 方法写入字段证据，且导出父组件 `inited()` 中存在唯一、未加工、无其他局部变量用途的同值 get/set 对；源 `created`、不明结构或有实际用途时不改写。规则不依赖 TaskManager 名称；目前仅对冻结的 TaskManager 回归候选验证过精确复现，不能声称已跨组件泛化成功。

新增边界测试；完整 221 项 unittest 通过。新批次 `formal_30_v4_task_manager_premount_guard_regression_v1` 仅复用冻结的 TaskManager 开发源样本，修复上限 6 轮（为此前五轮路径后允许一次四类残留定向修复，不保证通过）。配置 dry-run 通过；预运行证据冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_premount_guard_regression_v1_pre_run_20261029/`，31 个 payload SHA-256 校验通过，冻结时新输出、报告和 summary 路径均不存在。尚未发起模型请求；运行后应先冻结原始结果，再分析自动与人工结论。此批次不是独立正式测试证据。


## 2026-10-29：挂载前写入规则完整开发回归结果（自动通过，人工待验）

用户运行 `formal_30_v4_task_manager_premount_guard_regression_v1` 后，先将原始 summary、repair history、源 Vue 和初始／六轮 San 冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_premount_guard_regression_v1_TaskManager_automatic_pass_20261029_232535/`，14 个 payload 的 SHA-256 校验通过。1/1 完成，1/1 自动通过，无执行错误；修复 6 轮，生成 97,242 tokens、修复 145,866 tokens、总计 243,108。最终静态与可视化错误均为空，结构相似度 0.9889；交互评估为空。

逐轮复核显示父组件 `inited()` 均无目标挂载前同值回写，repair history 无 `removed_parent_premount_identity_writes` 标记，因此本轮不能归因于新增确定性规则实际触发。第 6 轮 San SHA-256 `409e6372bc2760fc08b43fe27c7f3c875867c873618296ad7e39d57536290a3e`，不同于前一人工通过候选 SHA-256 `dd80e7039e21827a205f0974ae3517510a51a05b70d10f33559e3056807f524b`。与旧候选相比，源事件回归门禁为 pass，无新增未解析源绑定或挂载前写入；但这不替代浏览器验收。

San 人工测试页只切换组件引用到本次冻结的第 6 轮产物；Vue 对照源码 SHA-256 与本次源 `ce3c3d4596792683e1e94a37764fdd542a97b80e4cff0c5ad8ba60c052f9e74c` 一致。等待用户复核设置／新建／编辑弹窗内部点击、遮罩关闭、拖拽排序、初始通知和自动同步，再记录人工结果。该批次复用开发样本，并非独立正式测试集证据。


## 2026-10-30：TaskManager 完整开发回归产物人工通过

用户在已切换到 `formal_30_v4_task_manager_premount_guard_regression_v1_TaskManager_automatic_pass_20261029_232535/repair_round_6.san` 的测试页人工复核后回复“通过”。本次确认绑定最终 San SHA-256 `409e6372bc2760fc08b43fe27c7f3c875867c873618296ad7e39d57536290a3e`，而非前一离线候选。此前请求复核设置／新建／编辑弹窗内部点击、遮罩关闭、拖拽排序、初始通知和自动同步；没有逐场景浏览器日志，不将其写成逐项测量结果。

人工通过证据冻结于 `data/experiments/snapshots/formal_30_v4_task_manager_premount_guard_manual_pass_20261030_141612/`，八个 payload SHA-256 校验通过，包括最终 San、Vue 源码、自动批次 summary／manifest 和两个测试页面。自动通过与人工通过现在都已记录，但该批次复用了 TaskManager 开发源样本，不是独立测试集结果；此次确定性规则也未实际触发，不能据此声称跨组件泛化效果。现有 `formal_30_v3` 正式协议已运行，不可事后改写其结果。
